# Kandinsky V2A — Event-Layer Pilot

## Question

For one short, busy scene, is it more useful to generate one complete soundtrack, or separate editable layers for the visible event and the background?

We will use one five-second CC0 city-walking video. Qwen proposes visible events and timestamps. You review the proposal, then Kandinsky generates:
- one full-scene audio track;
- one foreground-footsteps track;
- one background-city track.

All three Kandinsky calls use the same video window, checkpoint, and seed. The two layers are mixed after generation. This is a controlled product-workflow pilot, not model training.

## Important scope and limits

- The Qwen event list is a suggestion. Review and correct timestamps before using it.
- Qwen analyzes sampled video frames; it does not hear the source clip. Background ambience below is a user-added hypothesis, not a Qwen-verified sound.
- Kandinsky video-to-audio requires both video and a text prompt. It does not guarantee isolated stems; each generated layer may contain leakage from other sounds.
- We keep the same full scene video for overlapping layers. Cropping each sound into a separate video can remove context and is unsuitable when events overlap.
- The model generates five seconds at a time. These calls run sequentially on one GPU to avoid VRAM contention. Prior A100 measurements were about 10 minutes per Full Lite output, so allow roughly 30 minutes for three calls, plus setup.
- Qwen3-VL-2B model card lists Apache-2.0. The older Qwen2.5-VL-3B license file requests a separate license for commercial use. Recheck all model and dependency licenses before commercial deployment.

References:
- Kandinsky video-to-audio mode: https://github.com/kandinskylab/kandinsky-6/blob/main/docs/generation_modes.md
- Qwen3-VL video and timestamp capabilities: https://github.com/QwenLM/Qwen3-VL
- Qwen3-VL-2B model card and license: https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct
- Qwen2.5-VL-3B license terms: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/blob/main/LICENSE

## 1 — Check the Colab GPU and prepare folders

Use an A100 80 GB runtime. The notebook stores model caches on the Colab disk with the most available space.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, gc
from pathlib import Path
from datetime import datetime, timezone

print("Python:", sys.version.split()[0])
if sys.version_info < (3, 13):
    raise RuntimeError("Use the Python 3.13+ Colab runtime used by the Kandinsky pilot.")

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. Select an A100 runtime and reconnect.")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True
)
print("GPU:", gpu.stdout.strip())

roots = [Path("/content/local-scratch"), Path("/content")]
CACHE_ROOT = max((p for p in roots if p.exists()), key=lambda p: shutil.disk_usage(p).free)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024 ** 3)
print(f"Cache root: {CACHE_ROOT} | free: {free_gib:.1f} GiB")
if free_gib < 35:
    raise RuntimeError("At least 35 GiB free disk space is recommended.")

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["KANDINSKY_HOME"] = str(CACHE_ROOT / "kandinsky_home")

WORK = Path("/content/kandinsky_event_layers_pilot")
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "outputs"
for folder in (INPUT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, tail=3000):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-tail:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-6000:])
        raise RuntimeError("Command failed. See the output above.")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

## 2 — Download the traceable source clip and make one silent five-second window

The source is Wikimedia Commons' Big City Life clip. Its file page identifies the source and displays a CC0 1.0 dedication. The original is about 14 seconds. We use the first five seconds and remove its audio so Kandinsky must generate the soundtrack.

Source page: https://commons.wikimedia.org/wiki/File:Big_City_Life.webm

In [ ]:
from urllib.request import Request, urlopen

SOURCE_PAGE = "https://commons.wikimedia.org/wiki/File:Big_City_Life.webm"
SOURCE_URL = "https://upload.wikimedia.org/wikipedia/commons/2/2a/Big_City_Life.webm"
SOURCE_VIDEO = INPUT_DIR / "big_city_life.webm"
SILENT_VIDEO = INPUT_DIR / "city_walkers_00_05_silent.mp4"

if not SOURCE_VIDEO.is_file():
    request = Request(SOURCE_URL, headers={"User-Agent": "OrfeoV2APilot/1.0"})
    try:
        with urlopen(request, timeout=120) as response, open(SOURCE_VIDEO, "wb") as output:
            shutil.copyfileobj(response, output)
    except Exception as error:
        raise RuntimeError(
            f"Could not download the CC0 source automatically: {error}. "
            "Confirm network access or upload the source file from its Wikimedia Commons page."
        ) from error

run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(SOURCE_VIDEO), "-t", "5",
    "-map", "0:v:0", "-an",
    "-vf", "fps=24,scale='min(960,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
    "-pix_fmt", "yuv420p", str(SILENT_VIDEO)
])

if not SILENT_VIDEO.is_file():
    raise FileNotFoundError(f"Silent scene clip was not created: {SILENT_VIDEO}")

print("Source:", SOURCE_VIDEO.name, "| SHA-256:", sha256_file(SOURCE_VIDEO))
print("Scene window:", SILENT_VIDEO.name, "| SHA-256:", sha256_file(SILENT_VIDEO))
display(Video(filename=str(SILENT_VIDEO), embed=True))

## 3 — Ask Qwen for a visual event map

Qwen3-VL-2B analyzes the short video at two sampled frames per second. Ask it to return only events visibly supported by the frames, with approximate start/end times and confidence. It cannot confirm how an event sounds.

In [ ]:
import importlib.util

packages_needed = (
    importlib.util.find_spec("transformers") is None
    or importlib.util.find_spec("qwen_vl_utils") is None
    or importlib.util.find_spec("accelerate") is None
)
if packages_needed:
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.57.1", "accelerate", "qwen-vl-utils==0.0.14"
    ], tail=2000)

import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

QWEN_ID = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(QWEN_ID)
qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

analysis_request = """Inspect this five-second silent video.
Return valid JSON only, with this exact structure:
{
  "events": [
    {
      "id": "E1",
      "visible_action": "short description",
      "start_s": 0.0,
      "end_s": 1.0,
      "visual_evidence": "what is visible",
      "confidence": 0.0
    }
  ]
}
Include only visually supported actions that may need sound. Use approximate timestamps within 0 to 5 seconds. Do not invent audio, speech, music, materials, or off-screen events. If timing is uncertain, lower confidence. Return an empty events list if no clear action is visible."""

messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": SILENT_VIDEO.as_uri(), "fps": 2.0},
        {"type": "text", "text": analysis_request}
    ]
}]

qwen_text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
images, videos, video_kwargs = process_vision_info(
    messages, image_patch_size=16,
    return_video_kwargs=True, return_video_metadata=True
)
if videos is not None:
    videos, video_metadata = zip(*videos)
    videos, video_metadata = list(videos), list(video_metadata)
else:
    video_metadata = None

qwen_inputs = processor(
    text=qwen_text, images=images, videos=videos,
    video_metadata=video_metadata, padding=True,
    return_tensors="pt", do_resize=False, **video_kwargs
).to(qwen.device)

started = time.time()
qwen_ids = qwen.generate(**qwen_inputs, max_new_tokens=500, do_sample=False)
new_ids = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(qwen_inputs.input_ids, qwen_ids)
]
QWEN_EVENT_TEXT = processor.batch_decode(
    new_ids, skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]
QWEN_ANALYSIS_SECONDS = round(time.time() - started, 2)
print("Qwen analysis time:", QWEN_ANALYSIS_SECONDS, "seconds")
print(QWEN_EVENT_TEXT)

## 4 — Review and approve the event plan

Do not pass Qwen's timestamps straight to generation. Review them against the video. Edit the event times and prompts below. The city ambience layer is an explicit human-added hypothesis; Qwen cannot infer the actual sound from this silent video.

For this first comparison, both layers use the same full five-second video. This tests overlapping sounds without removing shared scene context.

In [ ]:
# Edit this event plan after reviewing QWEN_EVENT_TEXT.
# The background layer is human-added, not a sound detected by Qwen.
APPROVED_EVENTS = [
    {
        "id": "footsteps",
        "kind": "foreground",
        "start_s": 0.0,
        "end_s": 5.0,
        "prompt": "Soft repeated footsteps from several pedestrians walking on a city pavement, naturally spaced, foreground. No music, no speech."
    },
    {
        "id": "urban_bed",
        "kind": "background",
        "start_s": 0.0,
        "end_s": 5.0,
        "prompt": "Quiet distant city street ambience with diffuse pedestrian movement and a soft traffic wash. No prominent footsteps, no music, no intelligible speech."
    }
]

FULL_SCENE_PROMPT = (
    "Several pedestrians walking on a city pavement, with soft footsteps "
    "and distant urban movement. No music or speech."
)

print(json.dumps(APPROVED_EVENTS, ensure_ascii=False, indent=2))
print("Full-scene baseline prompt:", FULL_SCENE_PROMPT)

## 5 — Release Qwen from GPU memory

The analysis model is no longer needed. Free its memory before loading Kandinsky.

In [ ]:
del qwen, processor, qwen_inputs, qwen_ids, images, videos, video_metadata
gc.collect()
torch.cuda.empty_cache()
print("Qwen released. GPU memory is available for Kandinsky.")

## 6 — Install the pinned Kandinsky runtime and download Full Lite

This uses the same Kandinsky source revision as the previous Full Lite quality comparison. We use PyTorch SDPA and skip optional SageAttention compilation.

In [ ]:
REPO = Path("/content/kandinsky-6")
PINNED_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"

if not (REPO / ".git").exists():
    run([
        "git", "clone", "--filter=blob:none",
        "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)
    ], tail=2500)

run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT], tail=1500)
run([sys.executable, "-m", "pip", "install", "-q", "uv"], tail=1500)

if not Path("/usr/local/bin/just").exists():
    run([
        "bash", "-lc",
        "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"
    ], tail=2000)

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
print(setup.stdout[-3500:])
if setup.returncode:
    print(setup.stderr[-6000:])
    raise RuntimeError("Kandinsky environment setup failed; inspect the output above.")

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Kandinsky source commit:", SOURCE_COMMIT)

run(["just", "download", "lite"], cwd=REPO, tail=2500)
print("Full Lite checkpoint is ready.")

## 7 — Generate the full mix and the two candidate layers

We vary only the prompt/layer role. The input video, model, seed, and duration stay fixed. Calls are sequential so the A100 does not have to hold multiple generation jobs at once.

In [ ]:
DEVICE_CONFIG = REPO / "event-layers-full-lite.yaml"
DEVICE_CONFIG.write_text(
    """checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
    encoding="utf-8"
)

SEED = 42
WINDOW_SECONDS = 5
GENERATION_JOBS = [
    {"id": "full_scene", "prompt": FULL_SCENE_PROMPT},
    *[{"id": event["id"], "prompt": event["prompt"]} for event in APPROVED_EVENTS]
]

CANDIDATES = []
for job in GENERATION_JOBS:
    output_mp4 = OUTPUT_DIR / f"{job['id']}.mp4"
    output_wav = OUTPUT_DIR / f"{job['id']}.wav"

    command = [
        "just", "generate", job["prompt"],
        "--config", str(DEVICE_CONFIG),
        "--attention-engine", "sdpa",
        "--video-file", str(SILENT_VIDEO),
        "--time-length", str(WINDOW_SECONDS),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]

    print(f"Generating {job['id']} with Full Lite...")
    started = time.time()
    run(command, cwd=REPO, tail=3000)
    generation_seconds = round(time.time() - started, 2)

    if not output_mp4.is_file():
        raise FileNotFoundError(f"Generated MP4 is missing: {output_mp4}")

    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100",
        "-c:a", "pcm_s16le", str(output_wav)
    ])
    if not output_wav.is_file():
        raise FileNotFoundError(f"Generated WAV is missing: {output_wav}")

    CANDIDATES.append({
        "id": job["id"],
        "prompt": job["prompt"],
        "seed": SEED,
        "generation_seconds": generation_seconds,
        "mp4": output_mp4,
        "wav": output_wav,
        "wav_sha256": sha256_file(output_wav),
    })
    print(f"Finished {job['id']} in {generation_seconds:.1f} seconds.")

## 8 — Mix the two layers

The mix is a simple, editable preview: footsteps at full level and the ambience bed at a lower level. It is not loudness-normalized. If either source contains sound leakage, it will remain audible and reveal whether the layers are genuinely separable.

In [ ]:
FOOTSTEPS_WAV = OUTPUT_DIR / "footsteps.wav"
URBAN_BED_WAV = OUTPUT_DIR / "urban_bed.wav"
LAYER_MIX_WAV = OUTPUT_DIR / "layer_mix.wav"

if not FOOTSTEPS_WAV.is_file() or not URBAN_BED_WAV.is_file():
    raise FileNotFoundError("Both event-layer WAVs must exist before mixing.")

BED_GAIN = 0.40
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(FOOTSTEPS_WAV), "-i", str(URBAN_BED_WAV),
    "-filter_complex",
    f"[0:a]volume=1.0[fg];[1:a]volume={BED_GAIN}[bg];"
    "[fg][bg]amix=inputs=2:duration=longest:normalize=0[mix]",
    "-map", "[mix]", "-c:a", "pcm_s16le", str(LAYER_MIX_WAV)
])
print("Layer mix saved:", LAYER_MIX_WAV)

## 9 — Listen and compare

Listen for:
- whether the footsteps-only output still contains city ambience;
- whether the background-only output adds footsteps or other prominent events;
- whether the combined layers sound clearer than the full-scene output;
- whether you can move or adjust one layer without changing the other;
- timing errors, duplicated ambience, noise, and unwanted music/speech.

In [ ]:
from IPython.display import Audio, Video, display, Markdown

display(Markdown("### Source video"))
display(Video(filename=str(SILENT_VIDEO), embed=True))

candidate_by_id = {item["id"]: item for item in CANDIDATES}
for candidate_id in ("full_scene", "footsteps", "urban_bed"):
    item = candidate_by_id[candidate_id]
    display(Markdown(
        f"### {candidate_id} — {item['generation_seconds']:.0f}s generation"
    ))
    display(Audio(filename=str(item["wav"])))

display(Markdown("### Layer mix — footsteps + reduced city bed"))
display(Audio(filename=str(LAYER_MIX_WAV)))

## 10 — Save the experiment report and download the small package

The package includes the five-second CC0 video reference, generated WAVs, the layer mix, and the report. It does not include model weights.

In [ ]:
REPORT = {
    "experiment": "kandinsky_v2a_event_layers_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_page": SOURCE_PAGE,
    "source_sha256": sha256_file(SOURCE_VIDEO),
    "silent_window": SILENT_VIDEO.name,
    "silent_window_sha256": sha256_file(SILENT_VIDEO),
    "qwen_model": QWEN_ID,
    "qwen_event_text": QWEN_EVENT_TEXT,
    "qwen_analysis_seconds": QWEN_ANALYSIS_SECONDS,
    "approved_event_plan": APPROVED_EVENTS,
    "kandinsky_source_commit": SOURCE_COMMIT,
    "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers",
    "gpu": gpu.stdout.strip(),
    "seed": SEED,
    "duration_seconds": WINDOW_SECONDS,
    "bed_gain_in_mix": BED_GAIN,
    "candidates": [
        {
            "id": item["id"],
            "prompt": item["prompt"],
            "generation_seconds": item["generation_seconds"],
            "wav": item["wav"].name,
            "wav_sha256": item["wav_sha256"],
            "scores": {
                "scene_match_1_to_5": None,
                "timing_1_to_5": None,
                "quality_1_to_5": None,
                "noise_1_to_5": None,
                "keep": None,
                "notes": ""
            }
        }
        for item in CANDIDATES
    ],
    "layer_mix": LAYER_MIX_WAV.name,
    "notes": (
        "The separate prompts are candidate layers, not guaranteed isolated stems. "
        "All generations use the same video window."
    )
}

REPORT_PATH = WORK / "event_layers_report.json"
REPORT_PATH.write_text(
    json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8"
)

ZIP_PATH = Path("/content/kandinsky_v2a_event_layers_pilot.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(SILENT_VIDEO, "reference/city_walkers_00_05_silent.mp4")
    archive.write(REPORT_PATH, "event_layers_report.json")
    for item in CANDIDATES:
        archive.write(item["wav"], "audio/" + item["wav"].name)
    archive.write(LAYER_MIX_WAV, "audio/layer_mix.wav")

print("Package:", ZIP_PATH, "|", round(ZIP_PATH.stat().st_size / 1e6, 1), "MB")
from google.colab import files
files.download(str(ZIP_PATH))